# Introduction à PySpark – Traitement CSV

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/1to1code/blob/main/pySpark/notebooks/02_traitement_csv.ipynb)

Notebook conçu pour **Google Colab**.  
On explore la lecture de CSV, les transformations et les agrégations avec PySpark.

> **SparkUI** : accessible après la cellule de setup via le lien affiché.


In [2]:
# Installation + setup (Colab)
!pip install -q pyspark findspark
!git clone --filter=blob:none --sparse https://github.com/tanouar/1to1code.git -q \
  && cd 1to1code && git sparse-checkout set pySpark -q

import os, sys
PYSPARK_TOOLS = os.path.join('/content', '1to1code', 'pySpark')
if PYSPARK_TOOLS not in sys.path:
    sys.path.insert(0, PYSPARK_TOOLS)

from sparktools import setup_colab, download_iris

spark, monitor, helper = setup_colab("PySpark - Traitement CSV")


fatal: destination path '1to1code' already exists and is not an empty directory.
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

✓ SparkUI ouverte → cliquez sur le lien ci-dessus
✓ Spark 4.0.2  |  App : PySpark - Traitement CSV
✓ Memory : 4g  |  Shuffle partitions : 8
✓ SparkJobMonitor et SparkHelper initialisés


## 1. Créer un DataFrame

On crée un DataFrame manuellement avec `createDataFrame`.  
→ SparkUI : pas de job déclenché (pas d'action ici).


In [3]:
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
df_test = spark.createDataFrame(data, ["nom", "age"])
df_test.show()


+-------+---+
|    nom|age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



## 2. Lire un CSV

`download_iris()` télécharge le dataset Iris depuis UCI.  
→ SparkUI : le `read.csv` déclenche un job de scan du fichier.


In [4]:
csv_path = download_iris()   # télécharge dans /content/iris.csv

df = spark.read.csv(csv_path, header=False, inferSchema=True)
noms_colonnes = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]
df = df.toDF(*noms_colonnes)
df.show(5)


⏳ Téléchargement Iris...
✓ Iris téléchargé → /content/iris.csv
+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|      class|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows


In [5]:
df.printSchema()
df.show(5)


root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- class: string (nullable = true)

+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|      class|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows


## 3. Explorer les données

`describe()` déclenche un job de calcul sur toutes les colonnes numériques.  
→ SparkUI : observez les **stages** de ce job.


In [6]:
monitor.execute_and_monitor(
    lambda: df.describe().show(),
    "Statistiques descriptives"
)



🔵 Statistiques descriptives
📌 Tracking ID: 84335dd9
+-------+------------------+-------------------+------------------+------------------+--------------+
|summary|      sepal_length|        sepal_width|      petal_length|       petal_width|         class|
+-------+------------------+-------------------+------------------+------------------+--------------+
|  count|               150|                150|               150|               150|           150|
|   mean| 5.843333333333335| 3.0540000000000007|3.7586666666666693|1.1986666666666672|          NULL|
| stddev|0.8280661279778637|0.43359431136217375| 1.764420419952262|0.7631607417008414|          NULL|
|    min|               4.3|                2.0|               1.0|               0.1|   Iris-setosa|
|    max|               7.9|                4.4|               6.9|               2.5|Iris-virginica|
+-------+------------------+-------------------+------------------+------------------+--------------+


✅ SUCCESS
⏱️  Durée: 3.08s


## 4. Agrégations

Regroupement par espèce avec `groupBy` + `agg`.  
→ SparkUI : observez le plan d'exécution dans l'onglet **SQL / DataFrame**.


In [7]:
from pyspark.sql.functions import avg, max, min, count

monitor.execute_and_monitor(
    lambda: df.groupBy("class")
        .agg(
            count("*").alias("nb"),
            avg("petal_length").alias("petal_length_avg"),
            max("petal_width").alias("petal_width_max")
        )
        .orderBy("petal_length_avg", ascending=False)
        .show(),
    "Agrégation par espèce"
)



🔵 Agrégation par espèce
📌 Tracking ID: bc2eb378
+---------------+---+----------------+---------------+
|          class| nb|petal_length_avg|petal_width_max|
+---------------+---+----------------+---------------+
| Iris-virginica| 50|           5.552|            2.5|
|Iris-versicolor| 50|            4.26|            1.8|
|    Iris-setosa| 50|           1.464|            0.6|
+---------------+---+----------------+---------------+


✅ SUCCESS
⏱️  Durée: 2.36s
📊 Spark Job ID(s): [9, 8]
📦 Stages: 3 | Tasks: 3
🌐 Spark UI: http://localhost:4040/jobs/


## 5. Gros volume (100 millions de lignes)

On génère 100 millions de lignes pour mesurer la puissance de Spark.  
→ SparkUI : observez le nombre de **Tasks** exécutées en parallèle.


In [8]:
from pyspark.sql.functions import col, rand, when

df_big = spark.range(0, 100_000_000).toDF("id") \
    .withColumn("valeur", (rand() * 1000).cast("int")) \
    .withColumn("categorie", when(col("valeur") < 300, "Faible")
                              .when(col("valeur") < 700, "Moyen")
                              .otherwise("Élevé"))

monitor.execute_and_monitor(lambda: df_big.count(), "Count 100M")
monitor.execute_and_monitor(
    lambda: df_big.groupBy("categorie").count().show(),
    "Agrégation 100M"
)
monitor.show_history()



🔵 Count 100M
📌 Tracking ID: dcd52a28

✅ SUCCESS
⏱️  Durée: 0.90s
📊 Spark Job ID(s): [11, 10]
📦 Stages: 3 | Tasks: 5
🌐 Spark UI: http://localhost:4040/jobs/

🔵 Agrégation 100M
📌 Tracking ID: 70143f31
+---------+--------+
|categorie|   count|
+---------+--------+
|   Faible|30003035|
|    Moyen|40001738|
|    Élevé|29995227|
+---------+--------+


✅ SUCCESS
⏱️  Durée: 7.82s
📊 Spark Job ID(s): [13, 12]
📦 Stages: 3 | Tasks: 5
🌐 Spark UI: http://localhost:4040/jobs/

📜 HISTORIQUE DES JOBS

📊 Statistiques globales:
   Total jobs: 4
   Réussis: 4
   Échoués: 0
   Durée totale: 14.17s


1. Statistiques descriptives
   Tracking ID: 84335dd9
   Spark Job IDs: [7, 6]
   Stages: 3 | Tasks: 3
   Durée: 3.08s
   Status: ✅ SUCCESS

2. Agrégation par espèce
   Tracking ID: bc2eb378
   Spark Job IDs: [9, 8]
   Stages: 3 | Tasks: 3
   Durée: 2.36s
   Status: ✅ SUCCESS

3. Count 100M
   Tracking ID: dcd52a28
   Spark Job IDs: [11, 10]
   Stages: 3 | Tasks: 5
   Durée: 0.90s
   Status: ✅ SUCCESS

4. Agré